# Residual Attention U-Net — Semantic Segmentation (30m Sentinel-2)

Clones the project from GitHub, installs dependencies, and runs full training + evaluation on Kaggle GPU.

> **Before running:** Set your GitHub repo URL in the cell below.

In [ ]:
# ── 1. CONFIG ──────────────────────────────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"  # <-- set this
REPO_DIR        = "/kaggle/working/segment"

# ── 2. CLONE ───────────────────────────────────────────────────────────────
import os
if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    os.system(f"git -C {REPO_DIR} pull")
else:
    print("Cloning repo...")
    os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
print("Done.")

In [ ]:
# ── 3. INSTALL DEPENDENCIES ────────────────────────────────────────────────
os.system(f"pip install -q -r {REPO_DIR}/requirements.txt")
print("Dependencies installed.")

In [ ]:
# ── 4. VERIFY GPU ──────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {gpus}")
if not gpus:
    print("WARNING: No GPU detected. Go to Settings → Accelerator → GPU T4 x2")

In [ ]:
# ── 5. RUN TRAINING + EVALUATION ───────────────────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, f"{REPO_DIR}/main.py",
    "--data_path",        f"{REPO_DIR}/dataset",
    "--patch_size",       "256",
    "--patch_step",       "128",      # 50% overlap → more training patches
    "--epochs",           "200",
    "--batch_size",       "16",        # safe for T4/P100 (16 GB VRAM)
    "--lr",               "1e-4",
    "--patience",         "30",
    "--model_save_path",  "/kaggle/working/residual_attention_unet_30m.keras",
    "--output_dir",       "/kaggle/working/results/30m_full_model",
    "--num_samples",      "8",
]

print("Starting training... (output streamed below)")
print("Command:", " ".join(cmd))
print("-" * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("-" * 70)
print(f"Training finished with exit code: {proc.returncode}")

In [ ]:
# ── 6. DISPLAY RESULTS ─────────────────────────────────────────────────────
from IPython.display import Image, display
import glob

output_dir = "/kaggle/working/results/30m_full_model"
plots = [
    "training_history.png",
    "per_class_iou.png",
    "all_metrics_chart.png",
    "confusion_matrix.png",
    "predictions.png",
    "class_legend.png",
]
for plot in plots:
    path = os.path.join(output_dir, plot)
    if os.path.exists(path):
        print(f"\n── {plot} ──")
        display(Image(path))
    else:
        print(f"[not found] {path}")

In [ ]:
# ── 7. PRINT METRICS TABLE ─────────────────────────────────────────────────
import csv

csv_path = "/kaggle/working/results/30m_full_model/evaluation_results.csv"
if os.path.exists(csv_path):
    with open(csv_path) as f:
        for row in csv.reader(f):
            print(",".join(row))
else:
    print("CSV not found — training may not have completed.")